<a href="https://colab.research.google.com/github/marborox07-dot/VSGAN-tensorrt-docker/blob/main/Python3.14%2BTorch2.13%2Bcu130%2BQWen%2Bflux.2kelin%2Bkrea2-FP8-BF16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 Step 1: Prepare System (Python 3.14 + Torch 2.13 + cu130) Fix node
import os, sys, threading, time, shutil
from IPython.display import clear_output

uv_installed = shutil.which("uv") or os.path.exists("/root/.cargo/bin/uv")

if not uv_installed:
    print('⏳ [1/4] Bootstrapping uv and repairing Python environment...')
    !curl -LsSf https://astral.sh/uv/install.sh | sh
else:
    print('⏳ [1/4] uv is already installed. Skipping bootstrap...')

os.environ["PATH"] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"

!sudo apt-get update -y
!sudo apt-get install -y software-properties-common
!sudo add-apt-repository ppa:deadsnakes/ppa -y
!sudo apt-get update -y
!sudo apt-get install -y aria2 python3.14 python3.14-dev python3.14-venv libpython3.14-dev curl

print('⏳ Forcing pip reinstallation for Python 3.14 to fix missing module...')
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.14 1
!sudo update-alternatives --set python3 /usr/bin/python3.14
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.14

print('\n⏳ [2/4] Installing PyTorch 2.13+, xformers, and Dependencies (via uv)...')
!uv pip install --system "setuptools<82"
!uv pip install --system torch>=2.13.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
!uv pip install --system xformers --index-url https://download.pytorch.org/whl/cu130
!uv pip install --system llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!uv pip install --system numpy ftfy regex tqdm huggingface_hub diffusers==0.30.3 transformer-lens==1.17.0 gguf opencv-python
!uv pip install --system sageattention==1.0.6 PyOpenGL-accelerate
print('\n⏳ [3/4] Installing ComfyUI and Core Dependencies...')
%cd /content
if not os.path.exists('ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

if os.path.exists('/content/ComfyUI/user'):
    !rm -rf /content/ComfyUI/user

KJ_NODES_PATH = "/content/ComfyUI/custom_nodes/ComfyUI-KJNodes"

%cd /content/ComfyUI
!uv pip install --system -r requirements.txt

if not os.path.exists(KJ_NODES_PATH):
    !git clone https://github.com/kijai/ComfyUI-KJNodes {KJ_NODES_PATH}
%cd {KJ_NODES_PATH}
!uv pip install --system -r requirements.txt

%cd /content/ComfyUI/custom_nodes
if not os.path.exists('ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git
%cd ComfyUI-GGUF
!uv pip install --system -r requirements.txt

%cd /content/ComfyUI/custom_nodes
if not os.path.exists('ComfyUI-Manager'):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

%cd /content/ComfyUI/custom_nodes
if not os.path.exists('ComfyUI-Simple-Prompt-Batcher'):
    !git clone https://github.com/ai-joe-git/ComfyUI-Simple-Prompt-Batcher.git

BATCHER_REQ = '/content/ComfyUI/custom_nodes/ComfyUI-Simple-Prompt-Batcher/requirements.txt'
if os.path.exists(BATCHER_REQ):
    !uv pip install --system -r {BATCHER_REQ}

print('\n⏳ [4/4] Installing Additional Custom Nodes...')
%cd /content/ComfyUI/custom_nodes

extra_nodes = [
    "https://github.com/rgthree/rgthree-comfy.git",
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    "https://github.com/yolain/ComfyUI-Easy-Use.git",
    "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git",
    "https://github.com/DemonGatanjieu/Anomalous_Model_Browser.git",
    "https://github.com/danielwolber-wood/ComfyUI-QuickResolutionSelector.git"
]

for repo in extra_nodes:
    repo_name = repo.split('/')[-1].replace('.git', '')
    if not os.path.exists(repo_name):
        print(f"Downloading {repo_name}...")
        !git clone {repo}

    req_file = f"/content/ComfyUI/custom_nodes/{repo_name}/requirements.txt"
    if os.path.exists(req_file):
        print(f"Installing dependencies for {repo_name}...")
        !uv pip install --system -r {req_file}

clear_output()
print('✅ System is ready (Python 3.14 + Torch 2.13+ + CUDA 13.0)!')

!python3 -c "import torch; import xformers; import cv2; print(f'Torch: {torch.__version__} | xformers: {xformers.__version__} | OpenCV: {cv2.__version__} | CUDA: {torch.version.cuda}')"

In [ ]:
# @title 📥 Step 2: สร้างโครงสร้างโฟลเดอร์และดาวน์โหลดโมเดล Flux2Klein9B-fp8-bf16-master (/F29b-bf16-master-turbo-v4)
import os
from google.colab import userdata
from IPython.display import clear_output

# --- เปิดใช้งาน API Key ---
HF_TOKEN = userdata.get('HF_TOKEN')
MODELS_DIR = "/content/ComfyUI/models"

print('⏳ [1/5] กำลังเตรียมโครงสร้างโฟลเดอร์ทั้งหมดให้ ComfyUI...')
# รวมโฟลเดอร์มาตรฐานทั้งหมดของ ComfyUI ไว้ใน List เดียว
sub_folders = [
    'checkpoints', 'clip', 'clip_vision', 'configs', 'controlnet',
    'diffusion_models', 'embeddings', 'gligen', 'hypernetworks',
    'loras', 'photomaker', 'style_models', 'text_encoders',
    'unet', 'upscale_models', 'vae', 'vae_approx'
]

for d in sub_folders:
    os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)

print('\n👻 [2/5] กำลังสร้างไฟล์ตัวหลอก (v1-5-pruned-emaonly-fp16.safetensors)...')
# สร้างไฟล์ว่างๆ หลอก ComfyUI ตามหน้าต่าง UI เริ่มต้น เพื่อไม่ให้ขึ้น Error: required model is missing
dummy_checkpoint_path = f"{MODELS_DIR}/checkpoints/v1-5-pruned-emaonly-fp16.safetensors"
with open(dummy_checkpoint_path, 'w') as f:
    pass

# สร้างฟังก์ชันช่วยดาวน์โหลดเพื่อแก้ปัญหา Bash มองไม่เห็นตัวแปร
def download_file(url, output_dir, output_name):
    cmd = f'aria2c --console-log-level=error -c -x 6 -s 6 -k 1M --header="Authorization: Bearer {HF_TOKEN}" -d "{output_dir}" -o "{output_name}" "{url}"'
    os.system(cmd)

print('\n🚀 [3/5] กำลังดาวน์โหลด Text Encoder (qwen-3-8b-flux-klein.safetensors)...')
# 📌 แก้ไข: ย้ายไปที่โฟลเดอร์ clip/ ตามที่ ComfyUI ต้องการ
download_file(
    url="https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/text_encoders/qwen-3-8b-flux-klein.safetensors",
    output_dir=f"{MODELS_DIR}/clip",
    output_name="qwen-3-8b-flux-klein.safetensors"
)

print('\n🚀 [4/5] กำลังดาวน์โหลด VAE (fulx2vae-master.safetensors)...')
# VAE อยู่ที่เดิมถูกต้องแล้ว
download_file(
    url="https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/vae/fulx2vae-master.safetensors",
    output_dir=f"{MODELS_DIR}/vae",
    output_name="fulx2vae-master.safetensors"
)

print('\n🚀 [5/5] กำลังดาวน์โหลด โมเดลหลัก (F29b-bf16-master-turbo_v4.safetensors)...')
# 📌 แก้ไข: ย้ายไปที่โฟลเดอร์ unet/ (สำหรับสถาปัตยกรรมโมเดล Flux)
download_file(
    url="https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/diffusion_models/F29b-bf16-master-turbo_v4.safetensors",
    output_dir=f"{MODELS_DIR}/diffusion_models",
    output_name="F29b-bf16-master-turbo_v4.safetensors"
)

print('\n🚀 [6/6] กำลังดาวน์โหลด LoRA (fulx2vbase-lora-master.safetensors)...')
# LoRA อยู่ที่เดิมถูกต้องแล้ว
download_file(
    url="https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/fulx2vbase-lora-master.safetensors",
    output_dir=f"{MODELS_DIR}/loras",
    output_name="fulx2vbase-lora-master.safetensors"
)
print('\n🚀 [6/6] กำลังดาวน์โหลด LoRA (fulx2vbase-lora-master.safetensors)...')
# LoRA อยู่ที่เดิมถูกต้องแล้ว
download_file(
    url="https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/flux-2-klein-9b_extracted_lora_rank_128-fp32.safetensors",
    output_dir=f"{MODELS_DIR}/loras",
    output_name="flux-2-klein-9b_extracted_lora_rank_128-fp32.safetensors"
)
print('\n🚀 [6/6] กำลังดาวน์โหลด LoRA (clip_vision_h.safetensors)...')
# LoRA อยู่ที่เดิมถูกต้องแล้ว
download_file(
    url="https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/clip_vision_h.safetensors",
    output_dir=f"{MODELS_DIR}/clip_vision",
    output_name="clip_vision_h.safetensors"
)
clear_output()
print('✅ โครงสร้างโฟลเดอร์ สร้างไฟล์หลอก และดาวน์โหลดไฟล์ทั้งหมดพร้อมใช้งานแล้ว! (ไฟล์อยู่ในตำแหน่งที่ถูกต้อง 100%)')

In [ ]:
# @title 📥 Step 2: Directory Setup and Model Download (Flux2Klein9B - Optimized 4 pipeline)(ล่าสุดAdd Textencode lv cfg 0.9-1.0)
# @markdown Select your preferred Diffusion Model and Text Encoder below.
main_model = "Flux2Klein_9B_v4BaseFp8.safetensors" # @param ["F29b-bf16-master-turbo_v4.safetensors","F29b-fp8-master-turbo_V4.safetensors","Flux2Klein9B-FP8-turbo.safetensors","Flux2Klein9B-bf16-turbo.safetensors","Flux2Klein9B_base-bf16_SNOFS_v1.4.safetensors","Flux2Klein9b_3.0_fp8-turbo.safetensors","Flux2Klein_9B_v4BaseFp8.safetensors","Flux2Klein_base-9b-kv.safetensors","Flux2Klein_base-bf16_master_v4.safetensors","Flux2Klein_master_base_v3.safetensors","Flux2Klein_v1_Fp8.safetensors","flux2_klein_9b_base_fp8_V3.safetensors","None"]
text_encoder = "qwen_3_8b_fp8mixed.safetensors" # @param ["Qwen3-VL-Embedding-8B_fp8.safetensors","Qwen3-VL-Embedding-8B.safetensors","qwen_3_8b_fp8mixed.safetensors","qwen-3-8b-flux-klein.safetensors","qwen_3_8b_bf16.safetensors","None"]

import os
from google.colab import userdata
from IPython.display import clear_output

HF_TOKEN = userdata.get('HF_TOKEN')
MODELS_DIR = "/content/ComfyUI/models"
AUTH_HEADER = f"Authorization: Bearer {HF_TOKEN}"

print("[1/4] Initializing ComfyUI directory structure...")
sub_folders = [
    'checkpoints', 'clip', 'clip_vision', 'configs', 'controlnet',
    'diffusion_models', 'embeddings', 'gligen', 'hypernetworks',
    'loras', 'photomaker', 'style_models', 'text_encoders',
    'unet', 'upscale_models', 'vae', 'vae_approx'
]

for d in sub_folders:
    os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)

print(f"\n[2/4] Downloading Text Encoder ({text_encoder}) & VAE...")
text_encoder_url = f"https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/text_encoders/{text_encoder}"
!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 20M \
    -d "{MODELS_DIR}/text_encoders" \
    -o "{text_encoder}" \
    "{text_encoder_url}"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 6 -s 6 -k 20M \
    -d "{MODELS_DIR}/vae" \
    -o "fulx2vae-master.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/vae/fulx2vae-master.safetensors"

print(f"\n[3/4] Downloading Diffusion Model ({main_model}) & CLIP Vision...")
main_model_url = f"https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/diffusion_models/{main_model}"
!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 20M \
    -d "{MODELS_DIR}/diffusion_models" \
    -o "{main_model}" \
    "{main_model_url}"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 10M \
    -d "{MODELS_DIR}/clip_vision" \
    -o "clip_vision_h.safetensors" \
    "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/clip_vision_h.safetensors"

print("\n[4/4] Downloading LoRA Models...")

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
    -d "{MODELS_DIR}/loras" \
    -o "fulx2vbase-lora-master.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/fulx2vbase-lora-master.safetensors"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
    -d "{MODELS_DIR}/loras" \
    -o "flux-2-klein-9b_extracted_lora_rank_128-fp32.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/flux-2-klein-9b_extracted_lora_rank_128-fp32.safetensors"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
    -d "{MODELS_DIR}/loras" \
    -o "Flux.2_Klein_9B_Speculum.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/Flux.2%20Klein_9B_Speculum.safetensors"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
    -d "{MODELS_DIR}/loras" \
    -o "FK_teeththroat.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/FK_teeththroat.safetensors"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
    -d "{MODELS_DIR}/loras" \
    -o "FK_mountedfellatio.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/FK_mountedfellatio.safetensors"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
    -d "{MODELS_DIR}/loras" \
    -o "blowgangthroat.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/blowgangthroat.safetensors"

!aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
    -d "{MODELS_DIR}/loras" \
    -o "Get_Pose3_Klein2_9B_Shurik.safetensors" \
    "https://huggingface.co/geceff/Flux2Klein9B-comfyui/resolve/main/loras/Get_Pose3_Klein2_9B_Shurik.safetensors"

# @title 🎭 Optional: Download Face & Head Swap Models (BFS)
# @markdown เลือกโมเดลที่ต้องการใช้งานด้านล่าง หากไม่ต้องการโหลดให้เลือก "None"
# @markdown ---

bfs_face_model = "None" # @param ["None", "bfs_face_v1_qwen_image_edit_2509.safetensors"]
bfs_head_model = "None" # @param ["None", "bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors", "bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors", "bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors", "bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors"]

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    AUTH_HEADER = f"Authorization: Bearer {HF_TOKEN}"
except:
    AUTH_HEADER = ""

TARGET_DIR = f"{MODELS_DIR}/loras"
os.makedirs(TARGET_DIR, exist_ok=True)

BFS_REPO = "https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/main"

print("🔍 Checking BFS Model Options...")

if bfs_face_model != "None":
    print(f"\n[+] Downloading Face Model: {bfs_face_model}")
    face_url = f"{BFS_REPO}/{bfs_face_model}"
    !aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
        -d "{TARGET_DIR}" \
        -o "{bfs_face_model}" \
        "{face_url}"
else:
    print("\n[-] Face Model: Skipped (None selected)")

if bfs_head_model != "None":
    print(f"\n[+] Downloading Head Model: {bfs_head_model}")
    head_url = f"{BFS_REPO}/{bfs_head_model}"
    !aria2c --header="{AUTH_HEADER}" --summary-interval=60 -c -x 4 -s 4 -k 50M \
        -d "{TARGET_DIR}" \
        -o "{bfs_head_model}" \
        "{head_url}"
else:
    print("\n[-] Head Model: Skipped (None selected)")

print("\n✅ BFS Process completed!")
clear_output()
print("✅ Download completed successfully!")
print(f"📌 Selected Diffusion Model: {main_model}")
print(f"📌 Selected Text Encoder: {text_encoder}")

✅ Download completed successfully!
📌 Selected Diffusion Model: Flux2Klein_9B_v4BaseFp8.safetensors
📌 Selected Text Encoder: qwen_3_8b_fp8mixed.safetensors


In [ ]:
# @title 📥 Step 2.1: HF High-Speed Downloader (Multi-Links)
# @markdown ### 📌 1. ช่องโหลดแบบกำหนดเอง (Custom - เลือกโฟลเดอร์และตั้งชื่อไฟล์ได้)
custom_url = "https://huggingface.co/geceff/Krea2TurboFp32/resolve/main/diffusion_models/pornmasterKrea2_v2TurboBF16.safetensors" # @param {type:"string"}
custom_filename = "" # @param {type:"string"}
target_sub_folder = "diffusion_models" # @param [ "checkpoints", "clip", "clip_vision", "configs", "controlnet", "diffusion_models", "embeddings", "gligen", "hypernetworks", "loras", "photomaker", "style_models", "text_encoders", "unet", "upscale_models", "vae", "vae_approx"]

# @markdown ---
# @markdown ### 📝 2. Text Encoder (ลงโฟลเดอร์ text_encoders อัตโนมัติ)
text_encoder_url = "https://huggingface.co/geceff/Krea2TurboFp32/resolve/main/text_encoders_comfy/qwen3vl_4b_fp8_scaled.safetensors" # @param {type:"string"}

# @markdown ---
# @markdown ### 🎨 3. LoRA (ลงโฟลเดอร์ loras อัตโนมัติ)
lora_url_1 = "https://huggingface.co/geceff/Krea2TurboFp32/resolve/main/loras/krea2_identity_edit_v1_2.safetensors" # @param {type:"string"}
lora_url_2 = "https://huggingface.co/geceff/Krea2TurboFp32/resolve/main/loras/krea2_identity_edit_v1_2_r64.safetensors" # @param {type:"string"}
lora_url_3 = "" # @param {type:"string"}

# @markdown ---
# @markdown ### ✨ 4. VAE (ลงโฟลเดอร์ vae อัตโนมัติ)
vae_url = "https://huggingface.co/geceff/Krea2TurboFp32/resolve/main/vae/krea2_mage_HDR_vae_fp32_comfy.safetensors" # @param {type:"string"}
vae_url_2 = "https://huggingface.co/geceff/Krea2TurboFp32/resolve/main/vae/Krea_2_bf16_vae.safetensors" # @param {type:"string"}

import os
import re
from google.colab import userdata
from huggingface_hub import hf_hub_download

# --- Setup High-Speed & Disable Xet for Stability ---
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
MODELS_DIR = "/content/ComfyUI/models"

# ลองดึง Token ของ HF ถ้ามี
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = None

def download_hf_file(url_input, target_folder, manual_filename=""):
    url = url_input.strip()
    if not url:
        return # ถ้าไม่ได้ใส่ลิงก์ ให้ข้ามไป

    clean_repo_id = url
    clean_filename = manual_filename.strip()

    # แยก Repo ID และ Path จาก URL โดยละเอียด
    if "huggingface.co" in url:
        # ดึง {user/repo} และ {subfolder/path/file}
        url_match = re.search(r"huggingface\.co/([^/]+/[^/]+)/(?:resolve|blob)/[^/]+/(.+)", url)
        if url_match:
            clean_repo_id = url_match.group(1)
            if not clean_filename:
                clean_filename = url_match.group(2).split("?")[0] # ป้องกันกรณีมี query string ต่อท้าย

    if clean_repo_id and clean_filename:
        print(f"🚀 Source: {clean_repo_id} (File: {clean_filename})")
        print(f"📂 Destination: {target_folder}/")

        dest_path = os.path.join(MODELS_DIR, target_folder)
        os.makedirs(dest_path, exist_ok=True)

        try:
            downloaded_file = hf_hub_download(
                repo_id=clean_repo_id,
                filename=clean_filename,
                local_dir=dest_path,
                local_dir_use_symlinks=False,
                token=HF_TOKEN
            )
            print(f"✅ Finished: {downloaded_file}\n")
        except Exception as e:
            print(f"❌ Error downloading {clean_filename}: {e}\n")
    else:
        print(f"⚠️ Invalid URL. Please provide a direct HuggingFace resolve URL.\n   -> URL: {url}\n")

# --- เริ่มการดาวน์โหลด ---

if custom_url:
    print("--- 📌 Processing Custom Link ---")
    download_hf_file(custom_url, target_sub_folder, custom_filename)

if text_encoder_url:
    print("--- 📝 Processing Text Encoder ---")
    download_hf_file(text_encoder_url, "text_encoders")

lora_urls = [lora_url_1, lora_url_2, lora_url_3]
for i, l_url in enumerate(lora_urls, 1):
    if l_url:
        print(f"--- 🎨 Processing LoRA {i} ---")
        download_hf_file(l_url, "loras")

if vae_url:
    print("--- ✨ Processing VAE 1 ---")
    download_hf_file(vae_url, "vae")

if vae_url_2:
    print("--- ✨ Processing VAE 2 ---")
    download_hf_file(vae_url_2, "vae")
clear_output()
print("🎉 All downloads completed!")

🎉 All downloads completed!


In [ ]:
# @title 🚀 Step 3: Launch ComfyUI (Select Tunnel Method)ปรับปรุงการแสดงlog
Tunnel_Method = "Ngrok" # @param ["Ngrok", "Localtunnel", "Cloudflare"]
Localtunnel_Subdomain = "comfyui-t4-manager" # @param {type:"string"}
Clear_Interval_Minutes = 20 # @param {type:"integer"}

import os
import time
import urllib.request
import json
import subprocess
import threading
import re
from google.colab import userdata
from IPython.display import clear_output
import datetime

PORT = 8188

print("🧹 1. Cleaning up existing processes and database locks...")
os.system(f"fuser -k {PORT}/tcp > /dev/null 2>&1")
os.system("pkill -9 -f main.py > /dev/null 2>&1")
os.system("pkill -9 -f localtunnel > /dev/null 2>&1")
os.system("pkill -9 -f lt > /dev/null 2>&1")
os.system("pkill -9 -f cloudflared > /dev/null 2>&1")
os.system("pkill -9 -f ngrok > /dev/null 2>&1")

db_dir = "/content/ComfyUI/user/"
if os.path.exists(db_dir):
    os.system(f"rm -rf {db_dir}*.db*")

os.chdir('/content')

tunnel_info = {"method": Tunnel_Method, "url": "", "ip": ""}

def print_connection_info():
    if tunnel_info["method"] == "Ngrok" or tunnel_info["method"] == "Cloudflare":
        print("\n" + "🌟" * 15)
        print(f"🔗 ComfyUI Link ({tunnel_info['method']}): {tunnel_info['url']}")
        print("🌟" * 15 + "\n")
    elif tunnel_info["method"] == "Localtunnel":
        print("="*60)
        print(f"🔑 1. Endpoint IP: {tunnel_info['ip']}")
        print(f"🔗 2. ComfyUI link: {tunnel_info['url']}")
        print("="*60)
        print("⚠️ IMPORTANT: Refresh webpage if UI is missing.")
        print("="*60 + "\n")

def start_ngrok():
    try:
        NGROK_TOKEN = userdata.get('NGROK_TOKEN')
    except userdata.SecretNotFoundError:
        NGROK_TOKEN = ""

    if not NGROK_TOKEN:
        print("\n❌ ERROR: 'NGROK_TOKEN' not found in Colab Secrets!")
        return False

    print('\n⏳ Preparing secure Ngrok connection...')
    if not os.path.exists('/content/ngrok'):
        os.system('wget -q -O /content/ngrok.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz')
        os.system('tar -xf /content/ngrok.tgz -C /content')
        os.system('rm /content/ngrok.tgz')
        os.system('chmod +x /content/ngrok')

    os.system(f'/content/ngrok config add-authtoken {NGROK_TOKEN}')
    subprocess.Popen(['/content/ngrok', 'http', str(PORT), '--log=stdout'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

    try:
        req = urllib.request.urlopen('http://localhost:4040/api/tunnels')
        res = json.loads(req.read())
        tunnel_info["url"] = res['tunnels'][0]['public_url']
        print_connection_info()
        return True
    except Exception:
        print(f"\n⚠️ Failed to retrieve Ngrok link. Please try again.")
        return False

def start_localtunnel():
    print('\n🛠️ 2. Installing and preparing Localtunnel...')
    os.system('npm install -g localtunnel > /dev/null 2>&1')

    print('🔎 3. Retrieving Endpoint IP to unlock Localtunnel...')
    try:
        tunnel_info["ip"] = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    except Exception:
        tunnel_info["ip"] = "Failed to retrieve IP, please try again"

    print('🌍 4. Tunneling via Localtunnel in the background...')
    subprocess.Popen(f"npx localtunnel --port {PORT} --subdomain {Localtunnel_Subdomain} > /dev/null 2>&1", shell=True)
    time.sleep(3)
    clear_output(wait=True)

    tunnel_info["url"] = f"https://{Localtunnel_Subdomain}.loca.lt"
    print_connection_info()
    return True

def start_cloudflare():
    print('\n⏳ Installing and preparing Cloudflare Quick Tunnel...')

    if not os.path.exists('/content/cloudflared'):
        os.system('wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
        os.system('chmod +x /content/cloudflared')

    if os.path.exists('/content/cf_tunnel.log'):
        os.remove('/content/cf_tunnel.log')

    os.system(f'nohup /content/cloudflared tunnel --url http://127.0.0.1:{PORT} > /content/cf_tunnel.log 2>&1 &')
    time.sleep(8)

    try:
        with open('/content/cf_tunnel.log', 'r') as f:
            log_data = f.read()

        links = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_data)

        if not links:
            print("⏳ Waiting for Cloudflare response for another 5 seconds...")
            time.sleep(5)
            with open('/content/cf_tunnel.log', 'r') as f:
                log_data = f.read()
            links = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_data)

        if links:
            tunnel_info["url"] = links[-1]
            print_connection_info()
            return True
        else:
            print("\n⚠️ Cloudflare link not found. Please try again.")
            return False
    except Exception as e:
        print(f"\n⚠️ Error retrieving Cloudflare link: {e}")
        return False

is_ready = False
if Tunnel_Method == "Ngrok":
    is_ready = start_ngrok()
elif Tunnel_Method == "Localtunnel":
    is_ready = start_localtunnel()
elif Tunnel_Method == "Cloudflare":
    is_ready = start_cloudflare()

if is_ready:
    print(f"\n📺 System is streaming ComfyUI logs. Output will be cleared every {Clear_Interval_Minutes} minutes to prevent freezing.\n")
    os.chdir('/content/ComfyUI')

    comfy_process = subprocess.Popen(
        ['python3', 'main.py', '--listen', '--port', str(PORT), '--preview-method', 'auto', '--dont-print-server'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    def print_logs():
        for line in comfy_process.stdout:
            print(line, end="", flush=True)

    log_thread = threading.Thread(target=print_logs, daemon=True)
    log_thread.start()

    clear_timer_seconds = Clear_Interval_Minutes * 60
    while True:
        time.sleep(clear_timer_seconds)
        clear_output(wait=True)
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] Output auto-cleared to prevent Colab freeze.\n")
        print_connection_info()

[22:50:55] Output auto-cleared to prevent Colab freeze.


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
🔗 ComfyUI Link (Ngrok): https://figurine-headphone-tingling.ngrok-free.dev
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟



In [ ]:
# @title 📥 Step 2: สร้างโครงสร้างโฟลเดอร์, ดาวน์โหลดโมเดล WAN2.2 และติดตั้ง Frame Interpolation
import os
from google.colab import userdata
from IPython.display import clear_output

# --- เปิดใช้งาน API Key ---
HF_TOKEN = userdata.get('HF_TOKEN')
MODELS_DIR = "/content/ComfyUI/models"

# ล้างโฟลเดอร์ที่เคยสร้างผิดตำแหน่งทิ้ง
!rm -rf "/content/{MODELS_DIR}"
!rm -rf "/content/vae_dir"
!rm -rf "/content/text_encoders_dir"

print('⏳ [1/5] กำลังเตรียมโครงสร้างโฟลเดอร์มาตรฐานเดียวกับ Flux Step...')
sub_folders = [
    'checkpoints', 'clip', 'clip_vision', 'configs', 'controlnet',
    'diffusion_models', 'embeddings', 'gligen', 'hypernetworks',
    'loras', 'photomaker', 'style_models', 'text_encoders',
    'unet', 'upscale_models', 'vae', 'audio_encoders'
]

for d in sub_folders:
    os.makedirs(f"{MODELS_DIR}/{d}", exist_ok=True)

# สร้างฟังก์ชันช่วยดาวน์โหลดตามโครงสร้างที่คุณต้องการ
def download_file(url, output_dir, output_name):
    dest_path = os.path.join(output_dir, output_name)
    if not os.path.exists(dest_path):
        print(f'🚀 กำลังดาวน์โหลด: {output_name} -> {output_dir}')
        cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M --header="Authorization: Bearer {HF_TOKEN}" -d "{output_dir}" -o "{output_name}" "{url}"'
        os.system(cmd)
    else:
        print(f'✅ พบไฟล์ {output_name} อยู่แล้ว')

print('\n🚀 [2/5] กำลังดาวน์โหลดโมเดล WAN2.2 ลงตามโครงสร้างหลัก...')

#Text Encoders
download_file(
   "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/text_encoders/nsfw_wan_umt5-xxl_fp8_scaled.safetensors",
  f"{MODELS_DIR}/text_encoders", "nsfw_wan_umt5-xxl_fp8_scaled.safetensors"
)

# VAE
download_file(
    "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan_2.1_bf16_vae.safetensors",
    f"{MODELS_DIR}/vae", "wan_2.1_bf16_vae.safetensors"
)

# Unet
download_file(
   "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_high_noise_14B_Q4KM_L.gguf",
    f"{MODELS_DIR}/unet", "/wan2.2_i2v_high_noise_14B_Q4KM_L.gguf"
)

download_file(
    "https://huggingface.co/geceff/Wan2.2-Custom-Models-GGUF/resolve/main/wan2.2_i2v_low_noise_14B_Q4KM_L.gguf",
    f"{MODELS_DIR}/unet", "/wan2.2_i2v_low_noise_14B_Q4KM_L.gguf"
)

# Clip Vision
download_file(
    "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/audio_encoders/wav2vec2_large_english_fp16.safetensors",
    f"{MODELS_DIR}/audio_encoders", "wav2vec2_large_english_fp16.safetensors"
)
clear_output()
print('\n✅ โครงสร้างเดิมเป๊ะ! ระบบพร้อมแล้วครับ')

print('\n🚀 [3/5] กำลังติดตั้ง FFmpeg...')
!apt -y install -qq ffmpeg

print('\n🚀 [4/5] กำลังติดตั้ง Practical-RIFE สำหรับการแทรกเฟรม...')
%cd /content
if not os.path.exists('/content/Practical-RIFE'):
    !git clone https://github.com/Isi-dev/Practical-RIFE
%cd /content/Practical-RIFE
!python3 -m pip install git+https://github.com/rk-exxec/scikit-video.git@numpy_deprecation
!mkdir -p /content/Practical-RIFE/train_log
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/IFNet_HDv3.py -O /content/Practical-RIFE/train_log/IFNet_HDv3.py
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/RIFE_HDv3.py -O /content/Practical-RIFE/train_log/RIFE_HDv3.py
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/refine.py -O /content/Practical-RIFE/train_log/refine.py
!wget -q https://huggingface.co/Isi99999/Frame_Interpolation_Models/resolve/main/4.25/train_log/flownet.pkl -O /content/Practical-RIFE/train_log/flownet.pkl

print('\n🚀 [5/5] กำลังติดตั้ง Node: ComfyUI-Frame-Interpolation (VFI)...')
%cd /content/ComfyUI/custom_nodes
if not os.path.exists('ComfyUI-Frame-Interpolation'):
    !git clone https://github.com/Fannovel16/ComfyUI-Frame-Interpolation
%cd ComfyUI-Frame-Interpolation
!python3 -m pip install -r requirements.txt

clear_output()
print('✅ โครงสร้างโฟลเดอร์, ไฟล์โมเดล WAN2.2, และระบบแทรกเฟรม (RIFE + VFI) พร้อมใช้งานแล้ว!')

In [ ]:
# @title 🧹 สคริปต์เคลียร์ไฟล์โมเดล (รักษาโครงสร้างโฟลเดอร์)
import os
import shutil

MODELS_DIR = "/content/ComfyUI/models"
sub_folders = [
    'checkpoints', 'clip', 'clip_vision', 'configs', 'controlnet',
    'diffusion_models', 'embeddings', 'gligen', 'hypernetworks',
    'loras', 'photomaker', 'style_models', 'text_encoders',
    'unet', 'upscale_models', 'vae', 'vae_approx'
]

print("🗑️ เริ่มกระบวนการเคลียร์ไฟล์...")

for folder in sub_folders:
    folder_path = os.path.join(MODELS_DIR, folder)

    # ตรวจสอบว่ามีโฟลเดอร์หลักอยู่หรือไม่
    if os.path.exists(folder_path):
        # วนลูปหาไฟล์และโฟลเดอร์ย่อยที่อยู่ด้านใน
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            try:
                # ถ้าเป็นไฟล์หรือ Symbolic Link ให้ลบทิ้ง
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                # ถ้ามีโฟลเดอร์ย่อยหลงอยู่ข้างใน ให้ลบทิ้งทั้งโฟลเดอร์ย่อยนั้น
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f"❌ เกิดข้อผิดพลาดในการลบ {file_path}: {e}")
    else:
        # หากไม่มีโฟลเดอร์หลักตั้งแต่แรก ให้สร้างขึ้นมาใหม่เพื่อรักษาโครงสร้าง
        os.makedirs(folder_path, exist_ok=True)

print("✅ ล้างข้อมูลภายในโฟลเดอร์ที่ระบุเรียบร้อยแล้ว โครงสร้างหลักยังอยู่ครบ!")